# Lab 6 - Bokeh Time Series Analysis: Air Passengers

Recreate Lab 5 analyses using Bokeh interactive plots.

In [ ]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool, Span, Legend, LegendItem
from bokeh.palettes import Blues9
from bokeh.layouts import column
from bokeh.io import output_notebook
import statsmodels.tsa.stattools as ts

output_notebook()

# Load and preprocess
df = pd.read_csv("./datasets/AirPassengersDates.csv")
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month.astype(str)
df['Day'] = df['Date'].dt.day
df['Day_Name'] = df['Date'].dt.day_name()
df.head()

## Plot 1 – Basic Time Series Line Chart (Bokeh)

In [ ]:
source = ColumnDataSource(df)

p1 = figure(title="Air Passengers Over Time", x_axis_type='datetime', height=400, width=800,
           tools='pan,wheel_zoom,box_zoom,reset,save')
p1.line('Date', '#Passengers', source=source, line_width=2)

hover1 = HoverTool(tooltips=[("Date", "@Date{%F}"), ("Passengers", "@{#Passengers}")],
                  formatters={'@Date': 'datetime'})
p1.add_tools(hover1)
p1.xaxis.axis_label = "Date"
p1.yaxis.axis_label = "Number of Passengers"

show(p1)

## Plot 2 – Aggregated Bar Chart (Bokeh)

In [ ]:
monthly = df.groupby('Month')['#Passengers'].sum().reset_index()
source_monthly = ColumnDataSource(monthly)

p2 = figure(title="Total Passengers per Month", x_range=list(monthly['Month']), height=400, width=800,
           tools='pan,wheel_zoom,reset')
p2.vbar(x='Month', top='#Passengers', width=0.8, source=source_monthly, line_color='white', fill_color=Blues9[4])

hover2 = HoverTool(tooltips=[("Month", "@Month"), ("Passengers", "@{#Passengers}")])
p2.add_tools(hover2)
p2.xaxis.axis_label = "Month"
p2.yaxis.axis_label = "Total Passengers"

show(p2)

## Plot 3 – Mean and Standard Deviation (Bokeh)

In [ ]:
mean_val = df['#Passengers'].mean()
std_val = df['#Passengers'].std()
source_mean = ColumnDataSource(df)

p3 = figure(title="Passengers with Mean and Standard Deviation", x_axis_type='datetime', height=400, width=800,
           tools='pan,wheel_zoom,reset')
p3.line('Date', '#Passengers', source=source_mean, line_width=2)

# Span lines
mean_span = Span(location=mean_val, dimension='width', line_color='red', line_dash='dashed', line_width=2)
upper_span = Span(location=mean_val + std_val, dimension='width', line_color='green', line_dash='dashed', line_width=2)
lower_span = Span(location=mean_val - std_val, dimension='width', line_color='green', line_dash='dashed', line_width=2)
p3.add_layout(mean_span)
p3.add_layout(upper_span)
p3.add_layout(lower_span)

hover3 = HoverTool(tooltips=[("Date", "@Date{%F}"), ("Passengers", "@{#Passengers}")],
                  formatters={'@Date': 'datetime'})
p3.add_tools(hover3)
p3.xaxis.axis_label = "Date"
p3.yaxis.axis_label = "#Passengers"

show(p3)

## Plot 4 – Outlier Detection (Bokeh)

In [ ]:
z_score = (df['#Passengers'] - mean_val) / std_val
df['z_score'] = z_score
outliers = df[np.abs(z_score) > 2]
source_all = ColumnDataSource(df)
source_out = ColumnDataSource(outliers)

p4 = figure(title="Air Passengers with Outliers", x_axis_type='datetime', height=400, width=800,
           tools='pan,wheel_zoom,reset')
p4.line('Date', '#Passengers', source=source_all, line_width=2, color='blue')
p4.circle('Date', '#Passengers', source=source_out, size=8, color='red', legend_label='Outliers')

hover4 = HoverTool(tooltips=[("Date", "@Date{%F}"), ("Passengers", "@{#Passengers}")],
                  formatters={'@Date': 'datetime'})
p4.add_tools(hover4)
p4.xaxis.axis_label = "Date"
p4.yaxis.axis_label = "#Passengers"

show(p4)

## Plot 5 – Upsampling (Bokeh)

In [ ]:
df_ts = df.set_index('Date')
daily = df_ts.resample('D').asfreq().interpolate(method='linear')

source_orig = ColumnDataSource(df_ts.reset_index())
source_daily = ColumnDataSource(daily.reset_index())

p5 = figure(title="Upsampling to Daily Frequency", x_axis_type='datetime', height=400, width=800,
           tools='pan,wheel_zoom,reset')
p5.line('Date', '#Passengers', source=source_orig, line_width=2, alpha=0.7, legend_label='Original')
p5.line('Date', '#Passengers', source=source_daily, line_width=2, line_dash='dashed', legend_label='Upsampled Daily')

hover5 = HoverTool(tooltips=[("Date", "@Date{%F}"), ("Passengers", "@{#Passengers}")],
                  formatters={'@Date': 'datetime'})
p5.add_tools(hover5)
p5.legend.click_policy = "hide"
p5.xaxis.axis_label = "Date"
p5.yaxis.axis_label = "#Passengers"

show(p5)

## Plot 6 – Downsampling (Bokeh)

In [ ]:
yearly = df_ts['#Passengers'].resample('Y').mean().reset_index()

source_orig_down = ColumnDataSource(df_ts.reset_index())
source_yearly = ColumnDataSource(yearly)

p6 = figure(title="Downsampling to Yearly Frequency", x_axis_type='datetime', height=400, width=800,
           tools='pan,wheel_zoom,reset')
p6.line('Date', '#Passengers', source=source_orig_down, line_width=1, alpha=0.5, legend_label='Original')
p6.circle('Date', '#Passengers', source=source_yearly, size=8, line_color='navy', fill_color='white', legend_label='Yearly Average')
p6.line('Date', '#Passengers', source=source_yearly, line_width=2, color='navy')

hover6 = HoverTool(tooltips=[("Date", "@Date{%F}"), ("Passengers", "@{#Passengers}")],
                  formatters={'@Date': 'datetime'})
p6.add_tools(hover6)
p6.legend.click_policy = "hide"
p6.xaxis.axis_label = "Date"
p6.yaxis.axis_label = "#Passengers"

show(p6)

## Plot 7 – Lag Analysis (Bokeh)

In [ ]:
df_lag = df_ts.copy()
df_lag['shift1'] = df_lag['#Passengers'].shift(1)
df_lag['tshift1'] = df_lag['#Passengers'].shift(periods=1, freq='MS')
df_lag = df_lag.reset_index()

source_lag = ColumnDataSource(df_lag)

p7 = figure(title="Shift vs tShift", x_axis_type='datetime', height=400, width=800,
           tools='pan,wheel_zoom,reset')
p7.line('Date', '#Passengers', source=source_lag, line_width=2, color='blue', legend_label='Original')
p7.line('Date', 'shift1', source=source_lag, line_width=2, color='orange', line_dash='dashed', legend_label='Shift(1)')
p7.line('Date', 'tshift1', source=source_lag, line_width=2, color='green', line_dash='dotted', legend_label='tShift(MS)')

hover7 = HoverTool(tooltips=[("Date", "@Date{%F}"), ("Original", "@{#Passengers}"), ("Shift", "@shift1"), ("tShift", "@tshift1")],
                  formatters={'@Date': 'datetime'})
p7.add_tools(hover7)
p7.legend.click_policy = "hide"
p7.xaxis.axis_label = "Date"
p7.yaxis.axis_label = "#Passengers"

show(p7)

## Plot 8 – Autocorrelation (Bokeh)

In [ ]:
# Compute ACF manually
passengers = df_ts['#Passengers'].dropna()
lags = range(31)
acf_vals = [passengers.autocorr(lag) for lag in lags]
acf_df = pd.DataFrame({'lag': lags, 'acf': acf_vals})
source_acf = ColumnDataSource(acf_df)

p8 = figure(title="Autocorrelation Function (ACF)", height=400, width=800,
           tools='pan,wheel_zoom,reset')
p8.vbar(x='lag', top='acf', width=0.6, source=source_acf, line_color='navy', fill_alpha=0.7)

zero_span = Span(location=0, dimension='width', line_color='black', line_dash='dashed', line_width=1)
p8.add_layout(zero_span)

hover8 = HoverTool(tooltips=[("Lag", "@lag"), ("Autocorrelation", "@acf{0.3f}")])
p8.add_tools(hover8)
p8.xaxis.axis_label = "Lag"
p8.yaxis.axis_label = "Autocorrelation"
p8.x_range.start = 0

show(p8)